# Submit batch job to subset and save SPEAR-HI25 heat fluxes

Var names to download:
- lwflx
- shflx
- swup_sfc
- swdn_sfc
- evap 

In [3]:
import os
base_dir = '/work5/stb/MHW-gfdl/SPEAR'
#nco_fnc_str = 'ncks -O -d xT,-98.0,-10.0 -d yT,0.0,31.0'
#nco_fnc_str = 'ncks -O -d lon,262.0,350.0 -d lat,0.0,31.0'
batch_script = f'{base_dir}/03_submit_batch_ncks_ncrcat_spear'
#batch_script = f'{base_dir}/03_check_for_fils'

var_dict = {'ocean':{'var_list':['SST'], 
                     'ncks_str':'ncks -O -d xT,-98.0,-10.0 -d yT,0.0,31.0'},
           'atmos':{'var_list':['lwflx', 'shflx', 'swup_sfc', 'swdn_sfc', 'evap'],
                     'ncks_str':'ncks -O -d lon, 262.0, 350.0 -d lat,0.0,31.0'}}
yr0 = 1991
yrn = 2100
outdir = base_dir

nfil = 0
for ens_m in range(30):
    ens_str = str(ens_m+1).zfill(2) # ensemble string - zero padded
    indir = f'{base_dir}/SPEAR_c384_OM4p25_Hist_SSP245_IC1991_R61_ens_{ens_str}'

    #for var_type in var_dict:
    for var_type in ['atmos']:
        # creating a single string with all the variables for bash to work with
        bash_list = ''
        for var_nm in var_dict[var_type]['var_list']:
            bash_list = f'{bash_list} {var_nm}'

        # bringing together the sbatch command
        sbatch_fnc_str = f'sbatch --partition=batch {batch_script} {indir} {ens_str} {yr0} {yrn} "{bash_list}"'
        sbatch_fnc_str = f"{sbatch_fnc_str} '{var_dict[var_type]['ncks_str']}' {indir}"
 
        nfil+=1
        #print(ncfile)
        #outfile = f'{indir}/{var_nm}_ens{ens_str}_{ens_yr}.nc'
        os.system(sbatch_fnc_str)

print(nfil)

Submitted batch job 53780354
Submitted batch job 53780355
Submitted batch job 53780356
Submitted batch job 53780357
Submitted batch job 53780358
Submitted batch job 53780359
Submitted batch job 53780360
Submitted batch job 53780361
Submitted batch job 53780362
Submitted batch job 53780363
Submitted batch job 53780364
Submitted batch job 53780365
Submitted batch job 53780366
Submitted batch job 53780367
Submitted batch job 53780368
Submitted batch job 53780369
Submitted batch job 53780370
Submitted batch job 53780371
Submitted batch job 53780372
Submitted batch job 53780373
Submitted batch job 53780374
Submitted batch job 53780375
Submitted batch job 53780376
Submitted batch job 53780377
Submitted batch job 53780378
Submitted batch job 53780379
Submitted batch job 53780380
Submitted batch job 53780381
Submitted batch job 53780382
Submitted batch job 53780383
30


In [ ]:
import xarray as xr

path ='/work5/stb/MHW-gfdl/SPEAR/SPEAR-HI/SPEAR_c384_OM4p25_Hist_SSP245_IC1991_R61_ens_01/SST_ens01_1981-2010.nc'
ds = xr.open_dataset(path, decode_timedelta=True, chunks={})

In [7]:
ds

<xarray.Dataset> Size: 205MB
Dimensions:     (time: 10957, yT: 53, xT: 88, nv: 2, xTe: 361, yTe: 321)
Coordinates:
  * time        (time) object 88kB 1981-01-01 12:00:00 ... 2010-12-31 12:00:00
  * yT          (yT) float64 424B -0.0 0.3636 0.7273 1.091 ... 28.72 29.59 30.46
  * xT          (xT) float64 704B -97.5 -96.5 -95.5 -94.5 ... -12.5 -11.5 -10.5
  * nv          (nv) float64 16B 1.0 2.0
  * xTe         (xTe) float64 3kB -300.0 -299.0 -298.0 -297.0 ... 58.0 59.0 60.0
  * yTe         (yTe) float64 3kB -77.88 -77.67 -77.45 ... 89.59 89.8 90.0
Data variables:
    SST         (time, yT, xT) float32 204MB dask.array<chunksize=(10957, 53, 88), meta=np.ndarray>
    average_DT  (time) timedelta64[ns] 88kB dask.array<chunksize=(10957,), meta=np.ndarray>
    average_T1  (time) datetime64[ns] 88kB dask.array<chunksize=(10957,), meta=np.ndarray>
    average_T2  (time) datetime64[ns] 88kB dask.array<chunksize=(10957,), meta=np.ndarray>
    time_bnds   (time, nv) timedelta64[ns] 175kB dask.array<chunksize=(10957, 2), meta=np.ndarray>
Attributes:
    filename:   ice_daily.19810101-19901231.SST.nc
    title:      SPEAR_c384_o1_Hist_AllForc_IC1921_Q66_ens_01
    grid_type:  regular
    grid_tile:  N/A
    history:    Thu Mar  5 08:31:46 2026: ncrcat SST_ens01_1981.nc SST_ens01_...
    NCO:        netCDF Operators version 5.3.3 (Homepage = http://nco.sf.net,...

In [9]:
#check if time dim is increasing monotonically
ds.time.diff("time").min()

<xarray.DataArray 'time' ()> Size: 8B
array(86400000000, dtype='timedelta64[us]')
Attributes:
    long_name:       time
    cartesian_axis:  T
    calendar_type:   JULIAN
    bounds:          time_bnds

In [10]:
ds.SST

<xarray.DataArray 'SST' (time: 10957, yT: 53, xT: 88)> Size: 204MB
dask.array<open_dataset-SST, shape=(10957, 53, 88), dtype=float32, chunksize=(10957, 53, 88), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) object 88kB 1981-01-01 12:00:00 ... 2010-12-31 12:00:00
  * yT       (yT) float64 424B -0.0 0.3636 0.7273 1.091 ... 28.72 29.59 30.46
  * xT       (xT) float64 704B -97.5 -96.5 -95.5 -94.5 ... -12.5 -11.5 -10.5
Attributes:
    long_name:      sea surface temperature
    units:          deg-C
    cell_methods:   time: mean
    time_avg_info:  average_T1,average_T2,average_DT